# Emailing Participants with SendGrid or SMTP2GO

This notebook is used to email participants with their API keys. This is not to be used by participants themselves, but rather by the workshop organizers to send out keys.

However, you can take a look at the code to see how bulk e-mailing is done using SendGrid or SMTP2GO.

SendGrid (from Twilio) and [SMTP2GO](https://www.smtp2go.com/) are services that allow you to send emails through an API.
You can use either provider to send emails to participants with their API keys.

You can sign up for a free account at [SendGrid](https://sendgrid.com/).

There are other services that allow you to send emails in bulk, such as [Mailgun](https://www.mailgun.com/) and [Amazon SES](https://aws.amazon.com/ses/).

In old days this was done using `smtplib` and `email` libraries, however, these days most e-mail providers have limits on how many emails you can send per day, so it is better to use a service that is designed for this purpose.
This notebook will show you how to use SendGrid or SMTP2GO to send emails to participants with their API keys.

In [ ]:
import os
import requests

# Keep API keys in environment variables and never print the keys themselves.
my_sendgrid_key = os.getenv("SENDGRID_API")
my_smtp2go_key = os.getenv("SMTP2GO_API_KEY")

# SMTP2GO has verified this sender address.
SELF_EMAIL = "valdis.saulespurens@lnb.lv"
DEFAULT_TEST_DESTINATION = "valdis.saulespurens@gmail.com"
sendgrid_sender_email = os.getenv("SENDGRID_FROM_EMAIL")
smtp2go_sender_email = os.getenv("SMTP2GO_FROM_EMAIL", SELF_EMAIL)

if my_sendgrid_key:
    print("SendGrid API key is set.")
if my_smtp2go_key:
    print("SMTP2GO API key is set.")
if not (my_sendgrid_key or my_smtp2go_key):
    print("No email API key found in the environment; pass a key directly when calling email_single().")


## Sending one test email

`email_single(provider, key, message, destination=DEFAULT_TEST_DESTINATION)` sends a plain-text test message to the supplied destination. The destination defaults to `valdis.saulespurens@gmail.com`. Provider names are case-insensitive and may be `SendGrid` or `SMTP2GO`. The function returns a small result dictionary without exposing the API key.

SMTP2GO uses the verified `valdis.saulespurens@lnb.lv` sender address by default. SendGrid continues to use `SENDGRID_FROM_EMAIL`. The participant loop uses SendGrid when its key is available and otherwise uses SMTP2GO; set `EMAIL_PROVIDER` explicitly to override that choice.


In [ ]:
SMTP2GO_SEND_URL = "https://api.smtp2go.com/v3/email/send"
TEST_EMAIL_SUBJECT = "BSSDH email provider test"


def _normalise_provider(provider):
    if not isinstance(provider, str):
        raise TypeError("provider must be a string")

    normalised = provider.strip().upper()
    if normalised not in {"SENDGRID", "SMTP2GO"}:
        raise ValueError("provider must be either 'SendGrid' or 'SMTP2GO'")
    return normalised


def _send_email(provider, key, to_email, subject, message):
    provider = _normalise_provider(provider)
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f"A non-empty {provider} API key is required")
    if not isinstance(message, str) or not message.strip():
        raise ValueError("message must be a non-empty string")

    if provider == "SENDGRID":
        try:
            from sendgrid import SendGridAPIClient
            from sendgrid.helpers.mail import Mail
        except ImportError as exc:
            raise ImportError("SendGrid support requires the sendgrid package") from exc

        if not sendgrid_sender_email:
            raise ValueError("SENDGRID_FROM_EMAIL environment variable not set")
        mail = Mail(
            from_email=sendgrid_sender_email,
            to_emails=to_email,
            subject=subject,
            plain_text_content=message,
        )
        response = SendGridAPIClient(key.strip()).send(mail)
        if not 200 <= response.status_code < 300:
            raise RuntimeError(f"SendGrid request failed with HTTP {response.status_code}")
        return {"provider": "SendGrid", "status_code": response.status_code}

    response = requests.post(
        SMTP2GO_SEND_URL,
        headers={
            "Accept": "application/json",
            "X-Smtp2go-Api-Key": key.strip(),
        },
        json={
            "sender": smtp2go_sender_email,
            "to": [to_email],
            "subject": subject,
            "text_body": message,
        },
        timeout=30,
    )
    try:
        payload = response.json()
    except ValueError:
        payload = {}

    response_data = payload.get("data", {}) if isinstance(payload, dict) else {}
    if not response.ok:
        detail = response_data.get("error") or response.reason
        raise RuntimeError(f"SMTP2GO request failed with HTTP {response.status_code}: {detail}")
    if response_data.get("failed", 0):
        raise RuntimeError(f"SMTP2GO did not accept the email: {response_data.get('failures', [])}")

    return {
        "provider": "SMTP2GO",
        "status_code": response.status_code,
        "email_id": response_data.get("email_id"),
    }


def email_single(provider, key, message, destination=DEFAULT_TEST_DESTINATION):
    if not isinstance(destination, str) or not destination.strip():
        raise ValueError("destination must be a non-empty string")
    destination = destination.strip()

    result = _send_email(
        provider=provider,
        key=key,
        to_email=destination,
        subject=TEST_EMAIL_SUBJECT,
        message=message,
    )
    print(
        f"Test email sent to {destination} with {result['provider']}. "
        f"HTTP status: {result['status_code']}"
    )
    return result


In [ ]:
# #  Run this call manually when you want to send a real test email.
# email_single(
#     provider="SMTP2GO",
#     key=my_smtp2go_key,
#     message="Hello this is a test",
#     destination="enter valid email here"
# )


## Emailing all participants from the 2026 provisioned-key CSV

`email_participants(provider, key, csv_path=DEFAULT_PARTICIPANTS_CSV, delay=0.2)` reads `temp/BSSDH_2026_provisioned_keys.csv` by default and sends the updated BSSDH 2026 message to each eligible participant. It sends only rows whose `status` is `success` and whose `email` and `api_key` fields are present.

Defining the functions below does not send any email. Invoke `email_participants(...)` yourself in a new cell after reviewing the source CSV, subject, and message.


In [ ]:
from pathlib import Path
import pandas as pd

DEFAULT_PARTICIPANTS_CSV = Path("temp") / "BSSDH_2026_provisioned_keys.csv"
REQUIRED_PARTICIPANT_COLUMNS = {"email", "api_key", "status"}


def _resolve_existing_workshop_path(file_path):
    path = Path(file_path).expanduser()
    candidates = [path] if path.is_absolute() else [
        base / path for base in (Path.cwd(), *Path.cwd().parents)
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    checked = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"Could not find {path}. Checked:\n{checked}")


def load_provisioned_participants(csv_path=DEFAULT_PARTICIPANTS_CSV):
    resolved_path = _resolve_existing_workshop_path(csv_path)
    data = pd.read_csv(
        resolved_path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    missing_columns = REQUIRED_PARTICIPANT_COLUMNS - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required participant columns: {sorted(missing_columns)}"
        )

    email_values = data["email"].str.strip()
    api_key_values = data["api_key"].str.strip()
    status_values = data["status"].str.strip().str.lower()
    eligible = data.loc[
        status_values.eq("success")
        & email_values.ne("")
        & api_key_values.ne("")
    ].copy()

    eligible["email"] = eligible["email"].str.strip()
    eligible["api_key"] = eligible["api_key"].str.strip()

    invalid_email_count = int(
        (
            ~eligible["email"].str.fullmatch(
                r"[^@\s]+@[^@\s]+\.[^@\s]+"
            )
        ).sum()
    )
    if invalid_email_count:
        raise ValueError(
            f"Found {invalid_email_count} eligible row(s) with invalid email addresses"
        )

    duplicate_count = int(
        eligible["email"].str.lower().duplicated(keep=False).sum()
    )
    if duplicate_count:
        raise ValueError(
            f"Found {duplicate_count} eligible row(s) with duplicate email addresses"
        )
    if eligible.empty:
        raise ValueError("No eligible participant rows were found")

    eligible.attrs["source_path"] = str(resolved_path)
    print(
        f"Loaded {len(eligible)} eligible participant(s) from {resolved_path.name}. "
        f"Skipped {len(data) - len(eligible)} ineligible row(s)."
    )
    return eligible


In [ ]:
PARTICIPANT_EMAIL_SUBJECT = (
    "Your OpenRouter API key for BSSDH 2026 - 6 August"
)
WORKSHOP_REPOSITORY_URL = (
    "https://github.com/LNB-DH/BSSDH_2026_LLM_API_workshop"
)
WORKSHOP_PROGRAMME_URL = (
    "https://www.digitalhumanities.lv/bssdh/2026/Programme/"
)


def build_participant_email(participant):
    api_key = str(participant["api_key"]).strip()
    limit_usd = str(participant.get("limit_usd", "")).strip()
    expires_at = str(participant.get("expires_at_utc", "")).strip()

    key_details = []
    if limit_usd:
        key_details.append(f"Spending limit: USD {limit_usd}")
    if expires_at:
        key_details.append(f"Expiry: {expires_at}")
    key_details_text = (
        "\n".join(key_details) + "\n\n" if key_details else ""
    )

    return (
        "Hello,\n\n"
        "Thank you for participating in the BSSDH 2026 workshop "
        "\"Using LLMs in Humanities Research via API.\"\n\n"
        "Your individual OpenRouter API key is:\n"
        f"{api_key}\n\n"
        f"{key_details_text}"
        "Please keep this key secure and do not share it with anyone. "
        "It is intended only for your participation in the workshop.\n\n"
        "Workshop schedule - Thursday, 6 August 2026:\n"
        "11:30-13:00, 14:00-15:30, and 15:40-17:10\n"
        "National Library of Latvia, Conference Centre (level -1)\n\n"
        f"Programme: {WORKSHOP_PROGRAMME_URL}\n"
        f"Workshop materials: {WORKSHOP_REPOSITORY_URL}\n\n"
        "Please keep this email available during the workshop, as you will "
        "need the key for the practical exercises.\n\n"
        "Best regards,\n"
        "Valdis Saulespur\u0113ns\n"
        "BSSDH 2026 workshop instructor\n"
    )


In [ ]:
import time


def email_participants(
    provider,
    key,
    csv_path=DEFAULT_PARTICIPANTS_CSV,
    delay=0.2,
):
    provider = _normalise_provider(provider)
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f"A non-empty {provider} API key is required")
    if not isinstance(delay, (int, float)) or delay < 0:
        raise ValueError("delay must be a non-negative number")

    participants = load_provisioned_participants(csv_path)
    total_count = len(participants)
    sent_count = 0
    failures = []

    print(
        f"Sending {total_count} participant email(s) with {provider}; "
        f"delay: {delay} second(s)."
    )
    for participant_number, (_, participant) in enumerate(
        participants.iterrows(),
        start=1,
    ):
        try:
            result = _send_email(
                provider=provider,
                key=key,
                to_email=participant["email"],
                subject=PARTICIPANT_EMAIL_SUBJECT,
                message=build_participant_email(participant),
            )
            sent_count += 1
            print(
                f"Sent participant {participant_number}/{total_count}. "
                f"HTTP status: {result['status_code']}"
            )
        except Exception as exc:
            failures.append(
                {
                    "participant_number": participant_number,
                    "error_type": type(exc).__name__,
                }
            )
            print(
                f"Failed participant {participant_number}/{total_count}: "
                f"{type(exc).__name__}"
            )

        if participant_number < total_count and delay:
            time.sleep(delay)

    summary = {
        "provider": provider,
        "source_file": participants.attrs.get("source_path"),
        "eligible_count": total_count,
        "sent_count": sent_count,
        "failed_count": len(failures),
        "failures": failures,
    }
    print(
        f"All participant emails processed. "
        f"Sent: {sent_count}; failed: {len(failures)}."
    )
    return summary


In [ ]:
# Run this manually in a new cell after reviewing the CSV and message:

email_summary = email_participants(
    provider="SMTP2GO",
    key=my_smtp2go_key,
)
email_summary
